<a href="https://colab.research.google.com/github/rajilsaj/nasa-mosaics-project/blob/xgboost/notebooks/02_window_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

drive.mount('/content/drive')

# Paths
BASE_PATH = "/content/drive/MyDrive/2026/www/nasa-mosaics-project"

# Check if BASE_PATH exists
if not os.path.exists(BASE_PATH):
    print(f"Warning: BASE_PATH does not exist: {BASE_PATH}")
    print("Please ensure the path is correct in your Google Drive.")
else:
    print(f"BASE_PATH found: {BASE_PATH}")


SPLIT_DIR = f"{BASE_PATH}/data/splits"
OUTPUT_DIR = f"{BASE_PATH}/data/windows"

WINDOW_SIZE = 60

os.makedirs(OUTPUT_DIR, exist_ok=True)

Mounted at /content/drive
BASE_PATH found: /content/drive/MyDrive/2026/www/nasa-mosaics-project


In [2]:
split_name = "train"  # change to val or test when needed

ml_df = pd.read_csv(f"{SPLIT_DIR}/ml_{split_name}.csv")
jackson_df = pd.read_csv(f"{SPLIT_DIR}/jackson_{split_name}.csv")

print("ML samples:", len(ml_df))
print("Jackson events:", len(jackson_df))


ML samples: 2513117
Jackson events: 225


In [3]:
ml_df = ml_df.sort_values("SCLK").reset_index(drop=True)
jackson_df = jackson_df.sort_values("SCLK").reset_index(drop=True)


In [4]:
windows = []
window_id = 0

for _, event in jackson_df.iterrows():

    event_sclk = event["SCLK"]

    # Find matching position
    matches = ml_df[ml_df["SCLK"] == event_sclk]
    if matches.empty:
        continue

    event_idx = matches.index[0]

    # Find first precursor
    precursor_region = ml_df.loc[:event_idx]
    precursor_region = precursor_region[precursor_region["gt_detection_win"] == True]

    if precursor_region.empty:
        continue

    first_precursor_idx = precursor_region.index[0]

    start_idx = first_precursor_idx - WINDOW_SIZE
    end_idx = first_precursor_idx

    if start_idx < 0:
        continue

    window = ml_df.iloc[start_idx:end_idx].copy()

    if len(window) != WINDOW_SIZE:
        continue

    window["window_id"] = window_id
    window["label"] = 1
    window["event_sclk"] = event_sclk

    windows.append(window)
    window_id += 1


In [5]:
if windows:
    result_df = pd.concat(windows, ignore_index=True)
    result_df.to_csv(f"{OUTPUT_DIR}/{split_name}_windows.csv", index=False)

    print("Saved:", f"{split_name}_windows.csv")
    print("Positive windows extracted:", window_id)
else:
    print("No windows extracted.")


Saved: train_windows.csv
Positive windows extracted: 225
